In [1]:
import utils.dataset_generation as generate

In [2]:
generate.hello()

Hello world!


I have a cohort of new customers on e-commerce platform, and I need to forecast the future ltv of that cohort. I have historical data available, and I know that the customers can be clustered based on their behavior on that e-commerce platform into different type of customers, whose long term profitability varies greatly between the groups. Basically some customers will burn money on the platform quickly and then churn, others spend slowly but accumulate great amount of value in long term, some will spend only little and churn, etc. When all these customer groups are combined into one cohort and average cumulative value of that whole cohort is calculated, early on it is almost impossible to estimate superficially how big representation each different group has within that whole cohort by looking at the so far cumulated data, even though in long term the value between 2 cohorts might be very different depending on the distribution of the customers within the cohort. But if one could accurately assign each customer to a group based on their behavior on the platform with good enough confidence, one could actually predict pretty accurately the long term profitability of that cohort, since the long term pattern of each of these groups is reasonably well known from historical data.

The main problem here is to identify the attributes which are good predictors of where a customer belongs, and which attributes do not contribute to the prediction quality. To build the model of clustering the customers, a substantial historical dataset is available. Each user has unique id, and the session data for each user is available (so that attributes like how big carts they are buying, which product categories, how long they spend shopping, how long they spend exploring, what is the time between each session, how many shopping sessions they have done since first becoming customer etc. The individual attributes don't matter for this exercise, only the fact that they are available and can be turned into numerical data for the purpose of clustering).

The idea is to give weight to each of these attributes (the weighing factor for each attribute is currently unknown). If we would take these attributes and some initial weighs at the point of 30 days after first becoming customer (so each customer is a point in vector space defined by those weighed attributes), clustered the the customers based on their position in vector space using k-means clustering (k is unknown at first but we can use some initial value) and then mapped the ltv cumulation of each cluster up to 120 days (which is available in this historical data), and then applied the same weighing to a comparable dataset of customers at 30 days and applied the cumulative 120d prediction for those cluster of the first dataset to the second dataset, we can then measure how good prediction those weighs and the chosen value of k provided. We could then start varying each of these weighs and the value of k and see if the prediction power of that weighing improves or decreases.

By iterating long enough we could find ideal value of each weighing factor, and finally we would have as output a model with the power to predict the future 120d ltv of any cohort of customers at the point of 30 days, based on the used actions on the platform.

Is this method a known method in analytics? What is it called? Is it application of machine learning? Is there more information on the strengths and weaknesses of such an approach? Do you have initial suggestions on how to implement such an approach in python, what libraries to use and what kind of structure?

The method you described combines elements of clustering, regression, and optimization, and while it doesn't map directly to a named, single analytic method, it represents a known hybrid approach in predictive analytics and machine learning. Here's a breakdown of its components, methodology, and recommendations for implementation:

## 1. Methodology Overview
Clustering: Segmenting customers based on behavioral attributes using clustering methods like k-means. This is an unsupervised learning technique used to identify groups with similar behavior patterns.
Predictive Modeling: Assigning long-term customer lifetime value (LTV) to clusters by associating historical LTV data with each identified group.
Weight Optimization: Iteratively adjusting attribute weights (feature importance) and cluster count (k) to maximize prediction accuracy of cohort-level LTV.
## 2. Strengths of the Approach
Interpretability: By mapping behaviors to LTV predictions, stakeholders gain insights into key drivers of customer profitability.
Actionability: The approach allows targeted strategies for customer groups based on their LTV characteristics.
Scalability: With automated clustering and optimization, this method can be adapted to different customer cohorts over time.
## 3. Weaknesses of the Approach
Feature Engineering Sensitivity: Quality of predictions heavily depends on selecting and preprocessing the right behavioral attributes.
Cluster Instability: Clustering methods like k-means are sensitive to initialization, scaling, and the number of clusters (k).
Iterative Complexity: Optimizing weights and k can be computationally expensive, especially on large datasets.
Temporal Drift: Customer behaviors and LTV patterns may evolve, requiring frequent retraining and re-evaluation of the model.
## 4. Relation to Known Methods
This approach is an application of unsupervised learning (clustering) combined with supervised learning (LTV prediction) and optimization.
Related methodologies:
Customer Segmentation: A common use of clustering for marketing and customer analysis.
Weighted k-means Clustering: A variation of k-means where attributes are given different weights.
Bayesian Optimization: Often used for hyperparameter tuning and can be applied here to optimize weights and k.
Dimensionality Reduction + Clustering: Techniques like PCA or t-SNE might be used to preprocess high-dimensional data.
## 5. Libraries and Python Implementation
Here’s how you could implement this in Python:

Preprocessing
Use pandas for data handling.
Scale/normalize data using scikit-learn's StandardScaler or MinMaxScaler.
Engineer features (e.g., session behavior metrics) as needed.
Clustering
Use scikit-learn's KMeans or MiniBatchKMeans for clustering.
Optimize the number of clusters (k) using the elbow method or silhouette score.
Weight Optimization
Initialize weights randomly or equally and adjust iteratively.
Use scipy.optimize or Bayesian optimization libraries (e.g., optuna, hyperopt) to tune weights for better LTV prediction.
Prediction and Evaluation
Predict LTV for new cohorts using historical data as a reference.
Evaluate using metrics like mean squared error (MSE) or mean absolute error (MAE) between predicted and actual LTVs.

In [ ]:
# Implementation Outline
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

# 1. Load and preprocess data
data = pd.read_csv('customer_data.csv')
features = ['cart_size', 'session_duration', 'time_between_sessions', ...]
X = data[features].values

# 2. Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Initialize weights
weights = np.ones(X_scaled.shape[1])  # Equal weights initially

# 4. Weighted clustering function
def weighted_kmeans(X, weights, k):
    X_weighted = X * weights  # Apply weights
    kmeans = KMeans(n_clusters=k, random_state=42).fit(X_weighted)
    return kmeans.labels_

# 5. Objective function: Evaluate prediction accuracy
def objective_function(params):
    k, *weights = params
    k = int(round(k))  # Ensure k is an integer
    weights = np.array(weights)
    clusters = weighted_kmeans(X_scaled, weights, k)
    # Map historical LTV to clusters and compute predicted cohort LTV
    predicted_ltv = ...
    actual_ltv = ...
    return mean_squared_error(actual_ltv, predicted_ltv)

# 6. Optimize weights and k
initial_guess = [5] + list(weights)  # Start with k=5 and equal weights
bounds = [(2, 20)] + [(0, 1)] * len(weights)  # Range for k and weights
result = minimize(objective_function, initial_guess, bounds=bounds, method='L-BFGS-B')

# 7. Apply optimized weights and k for future predictions
optimal_k = int(result.x[0])
optimal_weights = result.x[1:]

## 6. Recommended Libraries
Core ML & Clustering: scikit-learn (k-means, scaling), pandas (data processing), numpy (numerical operations)
Optimization: scipy.optimize, optuna (for weight and k tuning)
Visualization: matplotlib or seaborn (visualizing clusters and results)


I have dataset of historical customer data. Basically it is describing lifetime value per customer cumulation by showing the total revenue generated by the customers, cumulating daily. So in that dataset there is running numer of day N from 0 to 189, and for each N the value is total revenue by all customers divided by the number of customers for whom we have been customer at least N days (so we have enough information).

What is the best way to approach in python applying this dataset in predicting the future ltv per customer for a cohor of customers which we have acquired just recently?

To predict the future lifetime value (LTV) per customer for a recently acquired cohort using your historical dataset, you can use time-series modeling or a combination of machine learning regression and trend analysis. Here's a structured approach to achieve this:

## 1. Approach Overview
Analyze Historical Data Trends:

Understand how LTV accumulates over time for historical cohorts.
Identify patterns like growth rates, saturation points, or decay.
Train a Predictive Model:

Use historical data to model the LTV growth trajectory.
Input features: Cohort attributes (e.g., acquisition size, average initial spend, etc.) and day N.
Output: Predicted cumulative LTV for a given day N.
Apply the Model:

Use the model to predict future LTV for the new cohort based on its current performance and historical patterns.
Validate and Improve:

Validate predictions with actual LTV values as they accumulate.
Adjust the model periodically to account for changes in customer behavior.
## 2. Suggested Methods
Method 1: Time-Series Extrapolation
If you have only the cumulative LTV per day (with no additional cohort attributes):

Fit a Growth Curve: Use curve fitting or a time-series forecasting method to model LTV growth.
Common growth models:
Logistic or Gompertz curve: Models saturation.
Polynomial regression: Captures non-linear trends.
Forecast Future Values: Use the fitted model to predict beyond observed days.
Method 2: Machine Learning Regression
If you have additional features describing cohorts or customer behavior:

Feature Engineering:
Include cohort-level features (e.g., acquisition size, average cart size, churn rate).
Include temporal features (e.g., day N).
Train a Model: Use regression models to predict cumulative LTV.
Models: Gradient Boosting (e.g., XGBoost, LightGBM), Random Forest, or Neural Networks.
Evaluate Performance: Use metrics like RMSE or MAE to assess prediction accuracy.

In [ ]:
# Implementation in Python
# Example: Time-Series Extrapolation

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt

# Load historical data
data = pd.read_csv('ltv_data.csv')  # Columns: 'day', 'cumulative_ltv'

# Define growth curve (e.g., logistic)
def logistic_curve(x, L, k, x0):
    return L / (1 + np.exp(-k * (x - x0)))

# Fit the curve
x_data = data['day']
y_data = data['cumulative_ltv']
popt, _ = curve_fit(logistic_curve, x_data, y_data, bounds=(0, [100, 1, 200]))

# Predict future values
future_days = np.arange(0, 300)
ltv_predictions = logistic_curve(future_days, *popt)

# Plot results
plt.plot(x_data, y_data, label='Historical Data')
plt.plot(future_days, ltv_predictions, label='Predicted LTV', linestyle='--')
plt.xlabel('Day')
plt.ylabel('Cumulative LTV')
plt.legend()
plt.show()

In [ ]:
# Example: Machine Learning Regression

from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# Load data
data = pd.read_csv('ltv_data_with_features.csv')  # Columns: 'day', 'feature_1', ..., 'cumulative_ltv'

# Prepare features and target
X = data[['day', 'feature_1', 'feature_2', ...]]
y = data['cumulative_ltv']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = GradientBoostingRegressor()
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse}")

# Predict future LTV
future_days = np.arange(0, 300).reshape(-1, 1)
future_features = ...  # Add cohort-level features for future data
future_X = np.hstack([future_days, future_features])
future_ltv = model.predict(future_X)

## 4. Choosing Between Methods
Use time-series extrapolation if your dataset has consistent LTV growth trends and no additional features.
Use machine learning regression if you have richer cohort-level features or customer behavioral data.
## 5. Key Considerations
Data Quality:

Ensure that the historical dataset covers diverse cohorts and customer behaviors.
Handle missing data and outliers effectively.
Model Validation:

Reserve a portion of historical data for validation to test prediction accuracy.
Temporal Consistency:

Ensure that the patterns in historical data remain relevant to the new cohort.
Regular Updates:

Re-train the model periodically as new customer data becomes available.
